# ✂️ ClipFactory.ai — Premium Shorts Engine
> OpusClip / Vizard.ai alternative — paste a YouTube URL, get viral 9:16 Shorts with face-tracking, Brand Kits, B-Roll, and Hormozi captions.

**Runtime:** Make sure you're on **T4 GPU** (Runtime > Change runtime type > T4 GPU)

### Steps
1. Run **Cell 0** to mount Google Drive (models persist across sessions)
2. Run **Cell 1** once per session (installs everything, ~3 min)
3. Run **Cell 2** to launch the app — open the public URL it prints
4. Paste a YouTube URL in the app UI and click Analyze

In [ ]:
# ── CELL 0: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted.')

In [ ]:
# ── CELL 1: Setup (run once per session, ~3 min) ───────────────────────────
import os, subprocess, sys

REPO_DIR = '/content/AI-Shorts-Generator-opus'

# Clone or update the repo
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/NiL4gh/AI-Shorts-Generator-opus',
                    REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Repo up to date.')

# Install Python dependencies
!pip install -q -r {REPO_DIR}/requirements.txt

# Install llama-cpp-python from prebuilt CUDA wheel (skips 10-min compile)
!pip install llama-cpp-python --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

# Install Deno for yt-dlp JS challenge solver (fixes YouTube bot-check)
!curl -fsSL https://deno.land/install.sh | sh
os.environ['PATH'] = os.path.expanduser('~/.deno/bin') + ':' + os.environ['PATH']

# Install fonts for captions
!apt-get install -q -y fonts-liberation 2>/dev/null || true

sys.path.insert(0, REPO_DIR)
print('✅ Ready to launch.')

In [ ]:
# ── CELL 2: Launch ClipFactory.ai ─────────────────────────────────────────────
# Open the printed gradio.live URL in your browser
import os, sys

REPO_DIR = '/content/AI-Shorts-Generator-opus'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Point models and output to Google Drive for persistence across sessions
os.environ['LLM_DIR']      = '/content/drive/MyDrive/clip_factory/models/llm'
os.environ['WHISPER_DIR']  = '/content/drive/MyDrive/clip_factory/models/whisper'
os.environ['OUTPUT_DIR']   = '/content/drive/MyDrive/clip_factory/output'
os.environ['PROJECTS_DIR'] = '/content/drive/MyDrive/clip_factory/projects'

# Create dirs if they don't exist
for d in [os.environ['LLM_DIR'], os.environ['WHISPER_DIR'],
          os.environ['OUTPUT_DIR'], os.environ['PROJECTS_DIR']]:
    os.makedirs(d, exist_ok=True)

!python app.py